In [1]:
from datetime import datetime
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
import requests
import json
import pandas as pd

API_TOKEN = 'ATATT3xFfGF0KIcaQjgj6yYFWfHHuftjwpCZyE4Fu05aXBEePGdvcYJKVOO3nGp9AbI_BbQfNYGJApuVKS6CoJLxR69-mkCzD3adXtzLfIHnAa1VcSs-vkha2W-DhRK0FgrK_StB5LPhhMyORjuOyWBeCTSNzm7sxVcXvUqTu4d3114mLFVoLYc=55620BF4'

# Setup

EMAIL = "margaret.woon@gsb.com.au"

BASE_URL = "https://cua.atlassian.net/wiki"
AUTH = HTTPBasicAuth(EMAIL, API_TOKEN)
# BASE_URL = "https://cua.atlassian.net/wiki/rest/api/content/"

## Replace the CONTENT_ID in the widget with the page we want to scrap

# page_name_variable = dbutils.widgets.get("PAGE_NAME_VARIABLE")
# content_id = dbutils.widgets.get("CONTENT_ID")
#url = 'https://cua.atlassian.net/wiki/spaces/dp1/pages/3080028473/System+User+Export+Data+Dictionary?exportView=true'
#url = f"https://cua.atlassian.net/wiki/rest/api/content/{CONTENT_ID}?expand=body.export_view"
# url_dp = f"https://cua.atlassian.net/wiki/spaces/DP/pages/{content_id}/{page_name_variable}"

#
# Replace the url here
#
#
Target_url = 'https://cua.atlassian.net/wiki/spaces/DP/pages/4253679922/Non-Voice+Reporting+Digital+Engagement+CDR'

last_one = Target_url.rfind("/")


PAGE_NAME_VARIABLE = Target_url[last_one+1:]

second_last = Target_url.rfind("/",0, last_one)

CONTENT_ID = Target_url[second_last + 1: last_one]

# print(f"PAGE_NAME_VARIABLE {PAGE_NAME_VARIABLE}")

# print(f"CONTENT_ID {CONTENT_ID}")

url_dp1 = f"{BASE_URL}/spaces/dp1/pages/{CONTENT_ID}/{PAGE_NAME_VARIABLE}"

url = f"{BASE_URL}/rest/api/content/{CONTENT_ID}?expand=body.export_view"
pages = []

# request the data from confluence

res = requests.get(url, auth=AUTH, verify=False)

# print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$")


output_file_name = PAGE_NAME_VARIABLE.replace("+","_")

with open(f"./output/{output_file_name}.txt", "w", encoding="utf-8") as file:
         file.write(res.text)
# Parse the HTML content with BeautifulSoup
html_content = BeautifulSoup(res.text, 'html.parser')

# Parse the HTML
soup = html_content

output_path = f"./output/soup_{output_file_name}.txt"
with open(output_path, "w", encoding="utf-8") as file:
         file.write(str(soup))

# Find all tables
tables = soup.find_all("tbody")

for idx, table in enumerate(soup.find_all("table"), start=1):
    headers = [th.get_text(strip=True) for th in table.find_all("th")]
    rows = []

    for row in table.find_all("tr"):
        cells = row.find_all("td")
        if not cells:
            continue  # skip empty rows
        rows.append([cell.get_text(separator='\n', strip=True) for cell in cells])


    print(f"{output_file_name} {idx} has {len(rows)} rows scrapped")

    if rows:
        print(f"Last row of table {idx} : {rows[-1]}")
        print(rows)

    if rows:
        df = pd.DataFrame(rows, columns=headers if headers and len(headers) == len(rows[0]) else None)
        print(df)
        csv_filename = f"./output/{output_file_name}_{idx}.csv"
        output_path = f"{csv_filename}"
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
        print(f"Extracted {output_file_name} {idx} → {csv_filename}")

C:\Users\miw\AppData\Roaming\Python\Python314\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cua.atlassian.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Non-Voice_Reporting_Digital_Engagement_CDR 1 has 5 rows scrapped
Last row of table 1 : ['', '', '', '']
[['1', 'Initial publish', 'Margaret Woon', ''], ['', '', '', ''], ['', '', '', ''], ['', '', '', ''], ['', '', '', '']]
  Version          Updates           Team Change Sign Off
0       1  Initial publish  Margaret Woon                
1                                                        
2                                                        
3                                                        
4                                                        
Extracted Non-Voice_Reporting_Digital_Engagement_CDR 1 → ./output/Non-Voice_Reporting_Digital_Engagement_CDR_1.csv
Non-Voice_Reporting_Digital_Engagement_CDR 2 has 12 rows scrapped
Last row of table 2 : ['']
[['Non_Voice_Reporting'], ['Non_Voice_Reporting_Hist'], [''], ['Digital_Engagement_CDR_yyyymmddThhmmss.csv'], [''], ['1'], ['hist'], [''], ['SCD not used- table is set to Append'], [''], [''], ['']]
                     